# Child, Learner, Machine: Benchmarking Real Language Models Against Human Mandarin Acquisition
##### *Author: Slavena Peneva-Kargiou*
##### *Course: SoftUni Deep Learning*
##### *Instructor: Yordan Darakchiev*
##### *Date: August 2026*

Builds on: [SoftUni Machine Learning Final Project](https://github.com/Slavena1/Softuni-Machine-Learning-Final-Project)

In [ ]:
%cd /content/Softuni-Deep-Learning-Final-Project/notebook

In [ ]:
import os
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from scipy.stats import spearmanr, mannwhitneyu

import torch

from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import KFold, cross_val_score
from sklearn.metrics import (
    mean_squared_error, mean_absolute_error, r2_score, classification_report
)

# LLM API clients — add once Section 4.5 is implemented (after the Aug 3 exercise)
# import anthropic
# import openai

## 1. Introduction

My Machine Learning project asked a simple question: when Mandarin speakers acquire new vocabulary - as children learning their first language, or as adults learning a second one, what makes a word easy or hard? The answer turned out to depend on who's doing the learning. Children lean on concreteness: words for things you can point at and picture come first. Adult learners lean on frequency: the words you hear most often stick first. A third group sat in that project too, but simply as an assumption rather than a measurement - "the language model" standing in as a kind of frequency-maximizing learner, represented just by a static word-frequency table pulled from the Chinese BabyLM corpus.

This project asks what happens if I stop assuming and start testing.

**The core idea:** take real language models, not just a frequency table, and ask them directly, the way you'd quiz a learner, which Mandarin words they seem to know well and which they stumble on. Then compare that pattern against real human data: which words Mandarin-speaking children learn first, and which words are ranked as harder on the HSK proficiency scale that adult learners study against. Do today's language models look more like the concreteness driven child, the frequency driven adult learner, or something else entirely? And does that answer change depending on *how* you ask - a small model's internal probabilities, or a large commercial model's actual behavior when prompted?

**Motivation:** while looking into the Chinese BabyLM Challenge's own evaluation pipeline - the resource my ML project drew its corpus from, I found that its three evaluation tracks (grammar, character structure, brain-signal alignment) do not include anything like an acquisition-order comparison, even though the general English language BabyLM pipeline has exactly this kind of task. This is something that I feel in a good position to try exploring, given that the ML project already built half the ingredients. I reached out to the team behind that pipeline in July 2026 to ask whether this seemed like a real gap worth exploring before committing further time to it (*no answer yet, fill in accordingly*).

**What is new here, versus the ML project:** that project used Ridge and Random Forest regression on three hand-picked features (frequency, concreteness, word length) to find the pattern in the first place. This project three additional things:
- replaces the frequency-table proxy for "the language model" with direct behavioral testing of real LLMs, via API;
- adds an actual neural architecture comparison - a dense network trained on pretrained transformer embeddings, using the same frozen-backbone transfer-learning pattern this course teaches for vision and language models, evaluated against the earlier Ridge/RF baseline rather than replacing it outright;
- treats reliability engineering (caching, retries, cost tracking) as part of the methodology, not as an afterthought.

The rest of this notebook is organized to make that comparison as fair and transparent as I can manage: the same data, the same train/ test discipline, and an explicit accounting of which results I am confident in and which are still open questions.



## 2. Related Work
The BabyLM Chalenge (Warstadt et al., 2023) anchors the practical part of this project. It asks what happens if a language model is trained on roughly the amount of text a child actually hears growing up. The English evaluation pipeline built for it includes an age-of-acquisition (AoA) prediction task: check whether a trained model's behavior on a word correlates with the age at which real children acquire it.

The 2026 BabyLM Workshop introduced a new multilingual track built on BabyBabelLM, a training data resource explicitly covering English, Dutch and Chinese (Jumelet et al., 2026), with the year's theme being "going beyond English" - a direct invitation for this kind of non-English acquisition work.  

The Chinese BabyLM Challenge, colocated with NLPCC 2026, extends this to Mandarin and it is where my ML project's corpus came from. Its own evaluation pipeline takes a different shape: grammatical acceptability (NLU), character-level structural knowledge (Hanzi), and alignment with human brain recordings (Cog), but nothing resembling the English pipeline's AoA-prediction task. That's the gap this project tries to fill - I checked this reading of their pipeline directly against their published evaluation code before committing to the idea.

## 3. Data

Same data as the ML project, reloaded here via the same loader functions (src/data_prep.py, unchanged).

In [ ]:
import sys
sys.path.insert(0, '/content/Softuni-Deep-Learning-Final-Project/src')

import data_prep as dp
import features as ft

DATA_DIR = '/content/Softuni-Deep-Learning-Final-Project/data'

df_aoa = dp.load_wordbank(f'{DATA_DIR}/wordbank_mandarin_items.csv')
hsk_paths = {level: f'{DATA_DIR}/hsk{level}.csv' for level in range(1, 7)}
df_hsk = dp.load_hsk(hsk_paths)
df_freq = dp.load_babylm_frequencies(
    f'{DATA_DIR}/babylm_zh_frequencies.csv',
    parquet_path=f'{DATA_DIR}/train-00000-of-00001.parquet'
)
df_xuli = dp.load_xuli_concreteness(
    f'{DATA_DIR}/Concretenss_Ratings_of_9877_Two_Character_Chinese_Words.xlsx')
df_liu = dp.load_liu_concreteness(f'{DATA_DIR}/liu_2007_single_char.txt')

print(f"Child AoA words: {len(df_aoa)}")
print(f"HSK words: {len(df_hsk)}")
print(f"Frequency table: {len(df_freq)} words")
print(f"Concreteness norms: {len(df_xuli)} (Xu & Li) + {len(df_liu)} (Liu et al.)")

In [ ]:
# Concreteness merging: Xu & Li (two-character) + Liu et al. (single-character)
df_xuli['concreteness_norm'] = dp.normalize_concreteness(df_xuli['concreteness'])
df_liu['concreteness_norm'] = dp.normalize_concreteness(df_liu['concreteness_raw'])

df_concrete_combined = pd.concat([
    df_xuli[['word', 'concreteness_norm']],
    df_liu[['word', 'concreteness_norm']]
], ignore_index=True)

# Reduplicated words (e.g. 妈妈, 爸爸): concreteness derived from the base character,
# since these carry no independent concreteness rating in either source dataset
liu_lookup = dict(zip(df_liu['word'], df_liu['concreteness_norm']))
missing_conc_words = df_aoa[~df_aoa['word'].isin(df_concrete_combined['word'])]['word'].tolist()
reduplications = [w for w in missing_conc_words if dp.is_reduplication(w)]

reduplication_rows = []
for word in reduplications:
    base_char = word[0]
    if base_char in liu_lookup:
        reduplication_rows.append({'word': word, 'concreteness_norm': liu_lookup[base_char]})
df_reduplications = pd.DataFrame(reduplication_rows)

df_concrete_final = pd.concat(
    [df_concrete_combined, df_reduplications], ignore_index=True
).drop_duplicates(subset='word', keep='first')

print(f"Final concreteness dataset: {len(df_concrete_final):,} unique words")

In [ ]:
df_child = dp.build_dataset(df_aoa.dropna(subset=['aoa']), 'aoa', df_freq, df_concrete_final)
df_adult = dp.build_dataset(df_hsk, 'hsk_level', df_freq, df_concrete_final)

FEATURES = ['log_frequency', 'concreteness', 'word_length']

df_child_analysis = df_child.dropna(subset=['aoa'] + FEATURES).copy().reset_index(drop=True)
df_adult_analysis = df_adult.dropna(subset=['hsk_level'] + FEATURES).copy().reset_index(drop=True)

print(f"Child analysis dataset: {len(df_child_analysis)} words")
print(f"Adult analysis dataset: {len(df_adult_analysis)} words")

In [ ]:
%%writefile /content/Softuni-Deep-Learning-Final-Project/src/neural_models.py
"""
neural_models.py

Architecture comparison, in the spirit of Ex.2 and Ex.4: don't just
train one model, compare a small progression of them on the same
train/test split used by baseline_models.py (Ridge/RF from the ML
project).

Comparison this supports:
  1. baseline_models.py - Ridge / RF on hand-picked features
     (frequency, concreteness, length) - Core
  2. dense NN on the same hand-picked features - isolates
     "did switching to a NN help, holding features constant?" (Stretch)
  3. dense NN on pretrained transformer embeddings (embeddings.py) -
     isolates "did switching features to embeddings help, holding
     architecture constant?" (Core)

The SAME train/test split (idx_*_train / idx_*_test from
baseline_models.split_raw) is reused across all three - this is the
leakage-adjacent lesson from the ML project resubmission, applied
here to keep the architecture comparison valid.
"""

import numpy as np
import torch
import torch.nn as nn
from sklearn.metrics import r2_score, mean_absolute_error, mean_squared_error
from sklearn.model_selection import train_test_split


def make_train_val_split(X_train, y_train, val_size=0.2, random_state=42):
    """
    Carve a validation set out of an existing training set, for
    early stopping during NN training. Call this on X_train/y_train
    only - never on X_test/y_test, which must stay untouched until
    final evaluation.
    """
    return train_test_split(X_train, y_train, test_size=val_size, random_state=random_state)


class DenseNet(nn.Module):
    """
    Small feed-forward network. Same architecture class used for
    both the hand-features (3-dim input) and embeddings (768-dim
    input) variants - only input_dim changes, so the comparison
    isolates the feature representation, not the model capacity.

    Kept deliberately small and regularized (dropout) given the
    small dataset size, especially for the high-dimensional
    embeddings case where overfitting risk is real.
    """

    def __init__(self, input_dim, hidden_dim=32, dropout=0.2):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(input_dim, hidden_dim),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(hidden_dim, hidden_dim // 2),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(hidden_dim // 2, 1),
        )

    def forward(self, x):
        return self.net(x).squeeze(-1)


def train_dense_nn(X_train, y_train, X_val, y_val, hidden_dim=32,
                    dropout=0.2, lr=1e-3, weight_decay=1e-4,
                    epochs=300, patience=20, batch_size=32, verbose=True):
    """
    Train a DenseNet with early stopping on validation loss.

    X_train/X_val should already be scaled (StandardScaler fit on
    the training portion only) before calling this - scaling is
    handled at the notebook level, same discipline as the Pipeline
    approach in baseline_models.py, not inside this function.

    Returns (model, history) where history has per-epoch train/val
    losses, for learning-curve plotting.
    """
    device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

    X_train_t = torch.tensor(np.asarray(X_train), dtype=torch.float32).to(device)
    y_train_t = torch.tensor(np.asarray(y_train), dtype=torch.float32).to(device)
    X_val_t = torch.tensor(np.asarray(X_val), dtype=torch.float32).to(device)
    y_val_t = torch.tensor(np.asarray(y_val), dtype=torch.float32).to(device)

    input_dim = X_train_t.shape[1]
    model = DenseNet(input_dim, hidden_dim, dropout).to(device)
    optimizer = torch.optim.Adam(model.parameters(), lr=lr, weight_decay=weight_decay)
    loss_fn = nn.MSELoss()

    n = X_train_t.shape[0]
    history = {'train_loss': [], 'val_loss': []}
    best_val_loss = float('inf')
    best_state = None
    patience_counter = 0

    for epoch in range(epochs):
        model.train()
        perm = torch.randperm(n, device=device)
        epoch_loss = 0.0
        for i in range(0, n, batch_size):
            idx = perm[i:i + batch_size]
            xb, yb = X_train_t[idx], y_train_t[idx]
            optimizer.zero_grad()
            pred = model(xb)
            loss = loss_fn(pred, yb)
            loss.backward()
            optimizer.step()
            epoch_loss += loss.item() * len(idx)
        epoch_loss /= n

        model.eval()
        with torch.no_grad():
            val_loss = loss_fn(model(X_val_t), y_val_t).item()

        history['train_loss'].append(epoch_loss)
        history['val_loss'].append(val_loss)

        if val_loss < best_val_loss - 1e-6:
            best_val_loss = val_loss
            best_state = {k: v.clone() for k, v in model.state_dict().items()}
            patience_counter = 0
        else:
            patience_counter += 1
            if patience_counter >= patience:
                if verbose:
                    print(f"Early stopping at epoch {epoch + 1} "
                          f"(best val loss: {best_val_loss:.4f})")
                break

    if best_state is not None:
        model.load_state_dict(best_state)

    return model, history


def evaluate_nn(model, X_test, y_test):
    """
    Return R2/MAE/RMSE - same metrics reported for Ridge in
    baseline_models.py, so results are directly comparable in one
    results table (Section 5).
    """
    device = next(model.parameters()).device
    model.eval()
    X_test_t = torch.tensor(np.asarray(X_test), dtype=torch.float32).to(device)
    with torch.no_grad():
        y_pred = model(X_test_t).cpu().numpy()

    return {
        'r2': r2_score(y_test, y_pred),
        'mae': mean_absolute_error(y_test, y_pred),
        'rmse': np.sqrt(mean_squared_error(y_test, y_pred)),
        'y_pred': y_pred,
    }

In [ ]:
%%writefile /content/Softuni-Deep-Learning-Final-Project/src/embeddings.py
"""
embeddings.py

Transfer learning component: extract word embeddings from a pretrained
Chinese transformer (frozen weights), for use as features in
neural_models.py - the same "frozen base + custom head" pattern taught
in the Vision Models exercise (Ex.6), applied to text, and consistent
with the Language Models lecture's own rule of thumb ("more training
data = less frozen layers") given our small word list stays fully
frozen.
"""

import numpy as np
import torch
from transformers import AutoModel, AutoTokenizer

_MODEL_CACHE = {}


def load_pretrained_model(model_name="hfl/chinese-macbert-base"):
    """
    Load a pretrained Chinese transformer + tokenizer, frozen
    (no fine-tuning - used purely as a fixed feature extractor).
    Cached in-memory by model_name so repeated calls don't re-load.
    """
    if model_name in _MODEL_CACHE:
        return _MODEL_CACHE[model_name]

    tokenizer = AutoTokenizer.from_pretrained(model_name)
    model = AutoModel.from_pretrained(model_name)
    model.eval()
    for param in model.parameters():
        param.requires_grad = False

    device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
    model = model.to(device)

    _MODEL_CACHE[model_name] = (model, tokenizer)
    return model, tokenizer


def get_word_embedding(word, model, tokenizer, pooling="mean"):
    """
    Get a single embedding vector for one Mandarin word, in isolation.

    pooling:
      'mean' - average sub-token hidden states, excluding [CLS]/[SEP]
      'cls'  - use the [CLS] token's hidden state
    """
    device = next(model.parameters()).device
    inputs = tokenizer(word, return_tensors="pt").to(device)

    with torch.no_grad():
        outputs = model(**inputs)

    hidden = outputs.last_hidden_state.squeeze(0)

    if pooling == "cls":
        vec = hidden[0]
    else:
        # mean pool over the actual word tokens (exclude [CLS] and [SEP])
        vec = hidden[1:-1].mean(dim=0) if hidden.shape[0] > 2 else hidden.mean(dim=0)

    return vec.cpu().numpy()


def build_embedding_matrix(word_list, model, tokenizer, pooling="mean", verbose=True):
    """
    Batch version of get_word_embedding. Returns (n_words, dim), in
    the same row order as word_list - important: this positional
    order is what lets the result be sliced with the same
    idx_train/idx_test used for the baseline models.
    """
    embeddings = []
    for i, word in enumerate(word_list):
        if verbose and i % 100 == 0:
            print(f"  {i}/{len(word_list)} words embedded...")
        embeddings.append(get_word_embedding(word, model, tokenizer, pooling=pooling))

    if verbose:
        print(f"  {len(word_list)}/{len(word_list)} done.")

    return np.vstack(embeddings)

## 4. Methods

### 4.1 Baseline: Ridge/ Random Forest (from the ML project)

In [ ]:
import baseline_models as bm

# Split on raw (unscaled) features - standardization happens inside each
# model's pipeline, fit on training data only
X_c_train, X_c_test, y_c_train, y_c_test, idx_c_train, idx_c_test = bm.split_raw(
    df_child_analysis, FEATURES, 'aoa')

X_a_train, X_a_test, y_a_train, y_a_test, idx_a_train, idx_a_test = bm.split_raw(
    df_adult_analysis, FEATURES, 'hsk_level')

print(f"Child train/test sizes: {len(X_c_train)} / {len(X_c_test)}")
print(f"Adult train/test sizes: {len(X_a_train)} / {len(X_a_test)}")

In [ ]:
param_grid = {'alpha': [0.001, 0.01, 0.1, 1, 10, 100]}
cv = KFold(n_splits=5, shuffle=True, random_state=42)

grid_child = bm.fit_ridge(X_c_train, y_c_train, param_grid, cv=cv)
ridge_child = grid_child.best_estimator_
y_c_pred = ridge_child.predict(X_c_test)
cv_scores_child = cross_val_score(ridge_child, X_c_train, y_c_train, cv=cv, scoring='r2')

print(f"Child AoA - Ridge (alpha={grid_child.best_params_['ridge__alpha']}):")
print(f"  R2: {r2_score(y_c_test, y_c_pred):.3f}  |  CV R2: {cv_scores_child.mean():.3f} +/- {cv_scores_child.std():.3f}")

grid_adult = bm.fit_ridge(X_a_train, y_a_train, param_grid, cv=cv)
ridge_adult = grid_adult.best_estimator_
y_a_pred = ridge_adult.predict(X_a_test)
cv_scores_adult = cross_val_score(ridge_adult, X_a_train, y_a_train, cv=cv, scoring='r2')

print(f"\nAdult HSK - Ridge (alpha={grid_adult.best_params_['ridge__alpha']}):")
print(f"  R2: {r2_score(y_a_test, y_a_pred):.3f}  |  CV R2: {cv_scores_adult.mean():.3f} +/- {cv_scores_adult.std():.3f}")

Random Forest is run as a **classifier**, not a regressor, on a binned version of each target — early/middle/late (via quantile split) for child AoA, and the six HSK levels directly for adult. This gives a discrete, model-native feature - importance view to compare against Ridge's coefficients in Section 5, alongside the continuous R² reported above.

In [ ]:
df_child_analysis['aoa_class'] = pd.qcut(df_child_analysis['aoa'], q=3, labels=['early', 'middle', 'late'])
X_c_tr_c, X_c_te_c, y_c_tr_c, y_c_te_c, idx_c_tr_c, idx_c_te_c = bm.split_raw(
    df_child_analysis, FEATURES, 'aoa_class', stratify_col='aoa_class')

rf_child = bm.fit_rf_classifier(X_c_tr_c, y_c_tr_c)
print("Child AoA - Random Forest (early/middle/late):")
print(classification_report(y_c_te_c, rf_child.predict(X_c_te_c)))

df_adult_analysis['hsk_class'] = df_adult_analysis['hsk_level'].astype(int)
X_a_tr_c, X_a_te_c, y_a_tr_c, y_a_te_c, idx_a_tr_c, idx_a_te_c = bm.split_raw(
    df_adult_analysis, FEATURES, 'hsk_class', stratify_col='hsk_class')

rf_adult = bm.fit_rf_classifier(X_a_tr_c, y_a_tr_c)
print("\nHSK Level - Random Forest (6 classes):")
print(classification_report(y_a_te_c, rf_adult.predict(X_a_te_c),
                             target_names=[f'HSK {i}' for i in range(1, 7)]))

In [ ]:
param_grid_lasso = {'alpha': [0.001, 0.005, 0.01, 0.05, 0.1, 0.5]}

lasso_child = bm.fit_lasso(X_c_train, y_c_train, param_grid_lasso, cv=cv)
lasso_adult = bm.fit_lasso(X_a_train, y_a_train, param_grid_lasso, cv=cv)

print(f"Lasso - Child (alpha={lasso_child.best_params_['lasso__alpha']}):")
for f_name, coef in zip(FEATURES, lasso_child.best_estimator_.named_steps['lasso'].coef_):
    print(f"  {f_name:<15}: {coef:>8.4f}  [{'RETAINED' if coef != 0 else 'ZEROED OUT'}]")

print(f"\nLasso - Adult (alpha={lasso_adult.best_params_['lasso__alpha']}):")
for f_name, coef in zip(FEATURES, lasso_adult.best_estimator_.named_steps['lasso'].coef_):
    print(f"  {f_name:<15}: {coef:>8.4f}  [{'RETAINED' if coef != 0 else 'ZEROED OUT'}]")

### 4.2 Dense NN on hand-picked features

In [ ]:
from sklearn.preprocessing import StandardScaler
import neural_models as nm

X_c_tr2, X_c_val, y_c_tr2, y_c_val = nm.make_train_val_split(X_c_train, y_c_train)
X_a_tr2, X_a_val, y_a_tr2, y_a_val = nm.make_train_val_split(X_a_train, y_a_train)

scaler_c = StandardScaler().fit(X_c_tr2)
scaler_a = StandardScaler().fit(X_a_tr2)

nn_child, hist_child = nm.train_dense_nn(
    scaler_c.transform(X_c_tr2), y_c_tr2, scaler_c.transform(X_c_val), y_c_val)
metrics_nn_child = nm.evaluate_nn(nn_child, scaler_c.transform(X_c_test), y_c_test)

nn_adult, hist_adult = nm.train_dense_nn(
    scaler_a.transform(X_a_tr2), y_a_tr2, scaler_a.transform(X_a_val), y_a_val)
metrics_nn_adult = nm.evaluate_nn(nn_adult, scaler_a.transform(X_a_test), y_a_test)

print(f"Child AoA  - Dense NN (hand-picked features): R2 = {metrics_nn_child['r2']:.3f}")
print(f"Adult HSK  - Dense NN (hand-picked features): R2 = {metrics_nn_adult['r2']:.3f}")

In [ ]:
import embeddings as emb

model, tokenizer = emb.load_pretrained_model("hfl/chinese-macbert-base")
print("MacBERT loaded and frozen.")

In [ ]:
# Build embeddings in the same order as the analysis dataframes
X_c_embeddings = emb.build_embedding_matrix(
    df_child_analysis['word'].tolist(),
    model, tokenizer, pooling="mean"
)

X_a_embeddings = emb.build_embedding_matrix(
    df_adult_analysis['word'].tolist(),
    model, tokenizer, pooling="mean"
)

print("Child embeddings shape:", X_c_embeddings.shape)
print("Adult embeddings shape:", X_a_embeddings.shape)

### 4.3 Transfer learning: pretrained transformer embeddings

In [ ]:
X_c_emb_train = X_c_embeddings[idx_c_train]
X_c_emb_test = X_c_embeddings[idx_c_test]
X_a_emb_train = X_a_embeddings[idx_a_train]
X_a_emb_test = X_a_embeddings[idx_a_test]

X_c_emb_tr2, X_c_emb_val, y_c_tr2b, y_c_valb = nm.make_train_val_split(X_c_emb_train, y_c_train)
X_a_emb_tr2, X_a_emb_val, y_a_tr2b, y_a_valb = nm.make_train_val_split(X_a_emb_train, y_a_train)

scaler_c_emb = StandardScaler().fit(X_c_emb_tr2)
scaler_a_emb = StandardScaler().fit(X_a_emb_tr2)

nn_child_emb, hist_child_emb = nm.train_dense_nn(
    scaler_c_emb.transform(X_c_emb_tr2), y_c_tr2b,
    scaler_c_emb.transform(X_c_emb_val), y_c_valb, dropout=0.4)
metrics_nn_child_emb = nm.evaluate_nn(nn_child_emb, scaler_c_emb.transform(X_c_emb_test), y_c_test)

nn_adult_emb, hist_adult_emb = nm.train_dense_nn(
    scaler_a_emb.transform(X_a_emb_tr2), y_a_tr2b,
    scaler_a_emb.transform(X_a_emb_val), y_a_valb, dropout=0.4)
metrics_nn_adult_emb = nm.evaluate_nn(nn_adult_emb, scaler_a_emb.transform(X_a_emb_test), y_a_test)

print(f"Child AoA  - Dense NN (MacBERT embeddings): R2 = {metrics_nn_child_emb['r2']:.3f}")
print(f"Adult HSK  - Dense NN (MacBERT embeddings): R2 = {metrics_nn_adult_emb['r2']:.3f}")

In [ ]:
!git status

In [ ]:
from google.colab import files
files.download('/content/Softuni-Deep-Learning-Final-Project/notebook/analysis.ipynb')

In [ ]:
!git status

In [ ]:
!git add src/neural_models.py src/embeddings.py notebook/analysis.ipynb

!git commit -m "Add neural_models.py, embeddings.py and update analysis notebook (Sections 3-4.3)"

!git push

In [ ]:
!git config --global user.email "peneva.s@gmail.com"
!git config --global user.name "Slavena1"

In [ ]:
!git add src/neural_models.py src/embeddings.py notebook/analysis.ipynb
!git commit -m "Add neural_models.py, embeddings.py and update analysis notebook (Sections 3-4.3)"
!git push

### 4.4 Attention weight analysis

???

### 4.5 LLM API testing: sentence completion + minimal pairs

### 4.6 Cost tracking

## 5. Results

## 6. Uncertainty Analysis

## 7. Error Analysis

## 8. Discussion

## 9. Limitations & Future Work

## 10. Conclusion

## 11. References

- Warstadt, A., et al. (2023). Findings of the BabyLM Challenge: Sample-efficient pretraining on developmentally plausible corpora. *Proceedings of the BabyLM Challenge*.
- Jumelet, J., et al. (2026). BabyBabelLM: A multilingual benchmark of developmentally plausible training data. *EACL 2026*. (verify full author list/formatting before final submission)
- Chinese BabyLM Challenge (2026). Co-located with NLPCC 2026. chinese-babylm.github.io


In [8]:
!rm -rf /content/Softuni-Deep-Learning-Final-Project

In [9]:
!git config --global user.email "peneva.s@gmail.com"
!git config --global user.name "Slavena1"

shell-init: error retrieving current directory: getcwd: cannot access parent directories: No such file or directory
fatal: Unable to read current working directory: No such file or directory
shell-init: error retrieving current directory: getcwd: cannot access parent directories: No such file or directory
fatal: Unable to read current working directory: No such file or directory


In [10]:
from google.colab import userdata
GITHUB_TOKEN = userdata.get('GITHUB_TOKEN')
! git clone https://{GITHUB_TOKEN}@github.com/Slavena1/Softuni-Deep-Learning-Final-Project.git /content/Softuni-Deep-Learning-Final-Project

shell-init: error retrieving current directory: getcwd: cannot access parent directories: No such file or directory
Cloning into '/content/Softuni-Deep-Learning-Final-Project'...
fatal: Unable to read current working directory: No such file or directory


In [ ]:
from google.colab import userdata
GITHUB_TOKEN = userdata.get('GITHUB_TOKEN')
! git remote set-url origin https://{GITHUB_TOKEN}@github.com/Slavena1/Softuni-Deep-Learning-Final-Project.git

In [ ]:
%cd /content/Softuni-Deep-Learning-Final-Project
!git add -A
!git commit -m "Sync notebook with latest work"
!git push

In [ ]:
!mv "/content/analysis.ipynb" /content/Softuni-Deep-Learning-Fin

In [1]:
from google.colab import userdata
GITHUB_TOKEN = userdata.get('GITHUB_TOKEN')
!git config --global user.email "peneva.s@gmail.com"
!git config --global user.name "Slavena1"
! git clone https://{GITHUB_TOKEN}@github.com/Slavena1/Softuni-Deep-Learning-Final-Project.git /content/Softuni-Deep-Learning-Final-Project

Cloning into '/content/Softuni-Deep-Learning-Final-Project'...
remote: Enumerating objects: 38, done.
remote: Counting objects: 100% (38/38), done.
remote: Compressing objects: 100% (29/29), done.
remote: Total 38 (delta 8), reused 25 (delta 3), pack-reused 0 (from 0)
Receiving objects: 100% (38/38), 20.97 KiB | 5.24 MiB/s, done.
Resolving deltas: 100% (8/8), done.


In [2]:
!mv /content/analysis.ipynb /content/Softuni-Deep-Learning-Final-Project/notebook/analysis.ipynb

mv: cannot stat '/content/analysis.ipynb': No such file or directory
